# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kratosontren/flyrank-ml-work/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
from datasets import load_dataset
import pandas as pd
from itertools import islice

daily = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True
)

sample = pd.DataFrame(list(islice(daily, 5000)))

print(sample.shape)
sample.head()

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

(5000, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


In [9]:
sample.columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events']

In [10]:
sample[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_pageviews",
        "ga4_sessions",
        "sessions_organic"
    ]
].describe()

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,sessions_organic
count,5000.000000,5000.000000,5000.000000,5000.0,5000.0,5000.0
mean,11.489400,0.105600,25.718199,0.0,0.0,0.0
std,18.728258,0.421526,23.504413,0.0,0.0,0.0
min,1.000000,0.000000,0.000000,0.0,0.0,0.0
25%,2.000000,0.000000,7.833333,0.0,0.0,0.0
50%,6.000000,0.000000,16.445906,0.0,0.0,0.0
75%,14.000000,0.000000,37.200000,0.0,0.0,0.0
max,424.000000,8.000000,127.000000,0.0,0.0,0.0


## 1. Method choice and why

# Method Choice

For this lane I selected **Logistic Regression**.

My lane is content refresh prioritization, where the objective is to identify pages that may require review based on historical search performance.

Logistic Regression was chosen because:

- it is simple and interpretable,
- it provides a transparent baseline machine learning model,
- it is less likely to overfit than more complex models on a small development sample,
- the model coefficients can be interpreted to understand which historical signals contribute most to the prediction.

The purpose of this notebook is to compare this model fairly against the Week-4 rule-based baseline using the same data and evaluation design.

In [11]:
feature_columns = [
    "gsc_impressions",
    "gsc_clicks"
]

features = sample[feature_columns].fillna(0)

# Proxy target
threshold = sample["gsc_avg_position"].median()

sample["target"] = (
    sample["gsc_avg_position"] > threshold
).astype(int)

print(sample["target"].value_counts())

features.head()

target
0    2500
1    2500
Name: count, dtype: int64


,gsc_impressions,gsc_clicks
0,30,0
1,5,0
2,1,0
3,6,0
4,5,0



# Split Design

An 80/20 train-test split is used.

Both the baseline rule and the Logistic Regression model are evaluated on the same test set.

Only historical variables available before the decision point are used.

No future performance windows or label-derived features are included.

This provides an honest comparison between the baseline rule and the machine learning model.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split

X = features
y = sample["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining target distribution")
print(y_train.value_counts())

print("\nTesting target distribution")
print(y_test.value_counts())

Training rows: 4000
Testing rows: 1000

Training target distribution
target
1    2000
0    2000
Name: count, dtype: int64

Testing target distribution
target
0    500
1    500
Name: count, dtype: int64


## 3. Train + compare vs my baseline
# Baseline vs Machine Learning Model

The baseline rule from Week 4 and the Logistic Regression model are evaluated using the same train-test split.

The same proxy target and evaluation metrics are used for both methods to ensure a fair comparison.

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ----------------------------
# Week-4 Baseline
# ----------------------------

baseline_score = (
    X_test["gsc_impressions"] / 100
    - X_test["gsc_clicks"]
)

baseline_pred = (
    baseline_score >
    baseline_score.median()
).astype(int)

# ----------------------------
# Logistic Regression
# ----------------------------

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

model_pred = model.predict(X_test)

# ----------------------------
# Comparison
# ----------------------------

comparison = pd.DataFrame({

    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],

    "Baseline": [

        accuracy_score(y_test, baseline_pred),

        precision_score(y_test, baseline_pred),

        recall_score(y_test, baseline_pred),

        f1_score(y_test, baseline_pred)

    ],

    "Logistic Regression": [

        accuracy_score(y_test, model_pred),

        precision_score(y_test, model_pred),

        recall_score(y_test, model_pred),

        f1_score(y_test, model_pred)

    ]

})

comparison

,Metric,Baseline,Logistic Regression
0,Accuracy,0.536000,0.552000
1,Precision,0.539301,0.528261
2,Recall,0.494000,0.972000
3,F1 Score,0.515658,0.684507


# Errors and Interpretation

The Logistic Regression model was evaluated on the same test split as the Week-4 baseline.

The model mainly relies on historical impressions and clicks.

Misclassifications may occur because:

- pages can experience seasonal search demand,
- click counts are sparse in this sample,
- the proxy target is an approximation of refresh need rather than a true label.

The results should be interpreted as observed and decision-support evidence rather than proof that refreshing a page will improve search performance.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
errors = X_test.copy()

errors["Actual"] = y_test.values
errors["Predicted"] = model_pred

misclassified = errors[
    errors["Actual"] != errors["Predicted"]
]

print("Misclassified rows:", len(misclassified))

misclassified.head(10)

Misclassified rows: 448


,gsc_impressions,gsc_clicks,Actual,Predicted
2656,13,0,0,1
3072,17,0,0,1
372,3,0,0,1
4496,12,0,0,1
3602,8,0,0,1
1358,12,0,0,1
329,8,0,0,1
2281,4,0,0,1
2596,1,0,0,1
2846,1,0,0,1


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit my repo URL on the card. Done.